In [1]:
import ee
from datetime import datetime, timezone

# 初始化（若未认证，取消下面两行的注释）
ee.Authenticate()
ee.Initialize()

C:\Users\pppmi\.conda\envs\oil\lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\pppmi\.conda\envs\oil\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [2]:
#对于保护区里每一个要素（feature）而言，遍历，然后输出每个保护区这个时间段存在sar图像的时间信息

# ========== 1. 加载保护区 FeatureCollection ==========
asset_path = 'projects/where-has-sar/assets/Navigator_AllSites_2604_final_keep'
fc = ee.FeatureCollection(asset_path)

#asset_path是 GEE 云端资产的路径（Asset ID）
#作用：读取已经上传到 GEE 云端 Assets 仓库里的数据，用`ee.FeatureCollection(asset_path)` / `ee.Image(asset_path)`加载。

total = fc.size().getInfo()


#fc.size() GEE 云端算子，向 GEE 服务器发起请求：统计这个 FeatureCollection 里面一共有多少个 Feature
#返回值类型：`ee.Number`（这还是**GEE 云端对象，不是 Python 数字**！）
#.getInfo()把云端`ee.Number` → Python 整数`int`


print(f"共加载 {total} 个保护区")

共加载 5069 个保护区


In [ ]:
#如何使用python直接上传本地文件进gee：（无需运行）
import geemap

local_shp = r"D:\gis_data\study_area.shp"
# 目标asset路径（上传之后的asset_path）
asset_path = "projects/ee-xxx/assets/study_area"

# 上传本地shp到GEE Asset
geemap.vector_to_asset(
    source=local_shp,
    destination=asset_path,
    overwrite=True
)
#上传是异步任务！提交代码后任务在 GEE 后台跑，需要等待任务完成，asset 才可以被读取。

In [ ]:
#形式A,直接导出到本地，适合数据量偏小的情况
import pandas as pd
# ========== 2. 定义时间范围：2015年1月 ==========
start_date = ee.Date('2015-01-01')
end_date   = ee.Date('2015-02-01')

# ========== 3. 为每个保护区计算有 SAR 图像的日期 ==========
def get_sar_dates(feature):
    """输入一个保护区 Feature，返回该保护区在 2015-01 有 SAR 图像的日期列表"""
    feature = ee.Feature(feature)
    region  = feature.geometry()

    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(region)                        # 相交即可
          .filterDate(start_date, end_date)
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .select('VV'))

    # 提取所有图像的时间戳
    timestamps = s1.aggregate_array('system:time_start')

    # 转换为日期字符串（yyyy-MM-dd），去重
    dates = timestamps.map(lambda ts: ee.Date(ts).format('YYYY-MM-dd'))
    unique_dates = dates.distinct()

    # 返回原 feature 加上日期列表字段
    return feature.set('sar_dates', unique_dates).set('image_count', s1.size())

# 用 map 批量处理（服务端）
result_fc = fc.map(get_sar_dates)

# ========== 4. 拉取结果为 Python 列表 ==========
# 注意：如果保护区数量很大（>1000），建议分批拉取或导出为 Drive 表
features_info = result_fc.getInfo()['features']

# ========== 5. 解析成 DataFrame ==========
rows = []
for feat in features_info:
    props = feat['properties']
    # 根据您的 Shapefile 属性字段调整，常见有 SITE_ID / NAME / site_name
    site_id   = props.get('SITE_ID') or props.get('site_id') or props.get('WDPAID') or props.get('wdpa_id')
    site_name = props.get('NAME') or props.get('site_name') or props.get('name')
    dates     = props.get('sar_dates') or []
    img_count = props.get('image_count', 0)

    if not dates:
        # 整个月无图像
        rows.append({
            'site_id': site_id,
            'site_name': site_name,
            'date': None,
            'image_count': 0
        })
    else:
        for d in dates:
            # 每个日期单独一行（同一天可能有多景，先记录总景数）
            # 如需按天细分景数，下面会说明
            rows.append({
                'site_id': site_id,
                'site_name': site_name,
                'date': d,
                'image_count': img_count  # 该保护区整月总景数（可改）
            })

df = pd.DataFrame(rows)
print(df.head())
print(f"总记录数：{len(df)}")

# ========== 6. 保存为 CSV ==========
df.to_csv('sar_2015_01_protected_areas.csv', index=False, encoding='utf-8-sig')
print("✅ 已保存为 sar_2015_01_protected_areas.csv")

In [3]:
#形式b，每个feature只给一条记录，导出在google里
import pandas as pd

# ========== 2. 定义时间范围：2015年1月 ==========
start_date = ee.Date('2015-01-01')
end_date   = ee.Date('2015-02-01')

# 为每个保护区生成一条记录（日期用分号拼接成一个字符串）
def make_row(feature):
    feature = ee.Feature(feature)
    region  = feature.geometry()

    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
          .filterBounds(region)
          .filterDate(start_date, end_date)
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .select('VV'))

    timestamps = s1.aggregate_array('system:time_start')
    dates = timestamps.map(lambda ts: ee.Date(ts).format('YYYY-MM-dd')).distinct()
    date_str = dates.join(';')   # 用分号拼成字符串

    return ee.Feature(None, {
        'site_id':    feature.get('SITE_ID'),
        'site_name':  feature.get('NAME'),
        'image_count': s1.size(),
        'dates':      date_str,
        'has_image':  s1.size().gt(0)
    })

export_fc = fc.map(make_row)

task = ee.batch.Export.table.toDrive(
    collection=export_fc,
    description='sar_2015_01_protected_areas',
    folder='GEE_exports',
    fileNamePrefix='sar_2015_01_protected_areas',
    fileFormat='CSV'
)
task.start()
print("✅ 导出任务已提交，请到 https://code.earthengine.google.com/tasks 查看进度")


✅ 导出任务已提交，请到 https://code.earthengine.google.com/tasks 查看进度


In [1]:
#通过形式A导出sar图像

import ee
from datetime import datetime, timezone

ee.Initialize()

# ========== 1. 加载研究区 ==========
asset_path = 'projects/where-has-sar/assets/test_poyhon'
fc = ee.FeatureCollection(asset_path)
region = fc.geometry()

# 小技巧：给 region 加一个极小的 buffer，避免几何边界导致的导出错误
region = region.buffer(10)   # 10 米缓冲，不影响实际范围

# ========== 2. 搜索条件 ==========
start_date = ee.Date('2015-01-01')
end_date   = ee.Date('2015-02-01')

s1 = (ee.ImageCollection('COPERNICUS/S1_GRD')
      .filterBounds(region)
      .filterDate(start_date, end_date)
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
      .select('VV'))

image_list = s1.toList(s1.size())
count = s1.size().getInfo()
print(f"共找到 {count} 景图像，准备提交导出任务...")

# ========== 3. 逐景提交导出任务 ==========
for i in range(count):
    img = ee.Image(image_list.get(i))

    img_id = img.get('system:id').getInfo()
    ts = img.get('system:time_start').getInfo()
    dt = datetime.fromtimestamp(ts / 1000, tz=timezone.utc)
    datetime_str = dt.strftime('%Y%m%d_%H%M%S')
    date_str = dt.strftime('%Y%m%d')

    safe_id = img_id.split('/')[-1]
    filename = f"{datetime_str}_{safe_id}"

    print(f"[{i+1}/{count}] 日期: {date_str} | ID: {img_id}")

    # 关键 1：clip 到研究区
    img_clip = img.clip(region)

    # 关键 2：使用图像原生投影（不写 crs）
    # 关键 3：fileDimensions 调大，避免分块
    task = ee.batch.Export.image.toDrive(
        image=img_clip,
        description=f'SAR_{datetime_str}_{i}',
        folder='GEE_SAR_Exports',
        fileNamePrefix=filename,
        region=region,
        scale=100,
        # crs 不写，使用图像原生投影
        maxPixels=1e13,
        fileDimensions=32768,          # 单个文件最大尺寸，尽可能大，避免分块
        fileFormat='GeoTIFF',
        formatOptions={'cloudOptimized': False}
    )
    task.start()

print("\n✅ 全部任务已提交！")
print("查看进度: https://code.earthengine.google.com/tasks")
print("下载文件: https://drive.google.com/ 进入 GEE_SAR_Exports")

C:\Users\pppmi\.conda\envs\oil\lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\pppmi\.conda\envs\oil\lib\site-packages\google\api_core\_python_version_support.py:255: FutureWarning: You are using a Python version (3.10.20) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


共找到 12 景图像，准备提交导出任务...
[1/12] 日期: 20150103 | ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SSV_20150103T004723_20150103T004752_004000_004D14_3FF4
[2/12] 日期: 20150104 | ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SSV_20150104T120216_20150104T120228_004021_004D87_AEFD
[3/12] 日期: 20150105 | ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SSV_20150105T124211_20150105T124239_004036_004DDB_0E90
[4/12] 日期: 20150107 | ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SSV_20150107T122622_20150107T122639_004065_004E8E_B9C6
[5/12] 日期: 20150109 | ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SSV_20150109T120954_20150109T121023_004094_004F47_B1D0
[6/12] 日期: 20150112 | ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SSV_20150112T123426_20150112T123438_004138_005042_4511
[7/12] 日期: 20150114 | ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SSV_20150114T121801_20150114T121833_004167_0050E1_48A5
[8/12] 日期: 20150115 | ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SSV_20150115T125804_20150115T125833_004182_005136_5CAA
[9/12] 日期: 20150120 | ID: COPERNICUS/S1_GRD/S1A_IW_GRDH_1SSV_20150120T005545_2015